# Usages

## Overview

- This page provides a general pipeline for analyzing Lipidomics (LiP) data using the `lipana` package.
- The example data used here is a truncated search report from DIA-NN. This dataset includes three experiment conditions, each with three replicates, and only 1000 proteins are retained.
- The example files are located in the `"path_to/LiPAna/example_data"` directory.

In [1]:
import gzip
import shutil
from pathlib import Path

workspace = Path(".").resolve().parents[1].joinpath("example_data")
print("current workspace:", str(workspace))
# Unzip gzipped files
for file in workspace.glob("*.gz"):
    with gzip.open(file, "rb") as f_in:
        with open(file.with_suffix(""), "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)


current workspace: /home/data/LiPAna/example_data


In [2]:
import sys

sys.path.append(str(workspace.parent))
import lipana

In [3]:
from time import sleep

import polars as pl

## Define the Experiment Layout

To annotate the expected report, it is essential to establish an experiment layout. Two methods for accomplishing this are introduced below:

1. Loading from a Text File: You can load a text file containing the experiment layout. An example of such a file might look like this ("replicate" column can be omitted):

|  run | condition | replicate |
| ---: | --------: | --------: |
| run1 |      ctrl |         1 |
| run2 |      ctrl |         2 |
| run3 |      ctrl |         3 |
| run4 |      exp1 |         1 |
| run5 |      exp1 |         2 |
| run6 |      exp1 |         3 |
| run7 |      exp2 |         1 |
| run8 |      exp2 |         2 |

In [4]:
import lipana

exp_layout = lipana.ExperimentLayout.from_file(workspace.joinpath("experiment_layout.txt"))
print("The input file is:")
exp_layout.exp_df

The input file is:


run,condition,replicate
str,str,i64
"""20240331_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",1
"""20240331_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",2
"""20240404_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",3
"""20240331_YSJ-HY_5_100-400ng_A_…","""H5_Y100""",1
"""20240331_YSJ-HY_5_100-400ng_B_…","""H5_Y100""",2
"""20240404_YSJ-HY_5_100-400ng_C_…","""H5_Y100""",3
"""20240331_YSJ-HY_25_100-400ng_A…","""H25_Y100""",1
"""20240331_YSJ-HY_25_100-400ng_B…","""H25_Y100""",2
"""20240404_YSJ-HY_25_100-400ng_C…","""H25_Y100""",3


2. Initializing with a Run-to-Condition Map: Alternatively, you can define the experiment layout by passing a dictionary mapping runs to their respective conditions. This approach is useful if you need to programmatically create or adjust the layout based on specific experiment parameters without relying on an external file

In [5]:
import lipana

exp_layout = lipana.ExperimentLayout.from_run_to_condition_map(
    {
        "20240331_YSJ-HY_2d5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-12_1_5154": "H2d5_Y100",
        "20240331_YSJ-HY_2d5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-13_1_5155": "H2d5_Y100",
        "20240404_YSJ-HY_2d5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-24_1_5206": "H2d5_Y100",
        "20240331_YSJ-HY_5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-14_1_5157": "H5_Y100",
        "20240331_YSJ-HY_5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-15_1_5158": "H5_Y100",
        "20240404_YSJ-HY_5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-28_1_5210": "H5_Y100",
        "20240331_YSJ-HY_25_100-400ng_A_90min_stdDIA-HomeMade_Slot1-19_1_5163": "H25_Y100",
        "20240331_YSJ-HY_25_100-400ng_B_90min_stdDIA-HomeMade_Slot1-20_1_5164": "H25_Y100",
        "20240404_YSJ-HY_25_100-400ng_C_90min_stdDIA-HomeMade_Slot1-34_1_5220": "H25_Y100",
    }
)

print("This results in the same experiment layout as the previous example, because replicate is incremented from 1")
exp_layout.exp_df

This results in the same experiment layout as the previous example, because replicate is incremented from 1


run,condition,replicate
str,str,i64
"""20240331_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",1
"""20240331_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",2
"""20240404_YSJ-HY_2d5_100-400ng_…","""H2d5_Y100""",3
"""20240331_YSJ-HY_5_100-400ng_A_…","""H5_Y100""",1
"""20240331_YSJ-HY_5_100-400ng_B_…","""H5_Y100""",2
"""20240404_YSJ-HY_5_100-400ng_C_…","""H5_Y100""",3
"""20240331_YSJ-HY_25_100-400ng_A…","""H25_Y100""",1
"""20240331_YSJ-HY_25_100-400ng_B…","""H25_Y100""",2
"""20240404_YSJ-HY_25_100-400ng_C…","""H25_Y100""",3


## Load a search report

- This step involves loading and annotating the search report
  - A DIA-NN report can originate from either running DIA-NN standalone or using FragPipe, which integrates DIA-NN for DIA data processing
  - For Spectronaut reports, certain columns are required, and a template for report settings can be generated using: `lipana.report.export_sn_report_setting("path_to_store_the_setting_file.rs")`

### First Load FASTA File(s)

- Before annotating the report, it is essential to load and parse FASTA files for species and sequence information
- `lipana.parse_fasta` will read one or more FASTA files and return a `ParsedFasta` object
  - `fasta_path` can be a path points to a FASTA or a list of paths (`fasta_path=some_path` or `fasta_path=[path1, path2]`)
  - `contam_fasta_path`: this parameter will be useful to strictly exclude potential contaminants. The species of any protein entries in the given file will be annotated as "Contam" instead of its original species, including those presented in `fasta_path`
  - `contaminations`: a list of additional protein entries can be given here, and this parameter can also work alone if `contam_fasta_path` is `None`; this parameter can also be used to define all the entries that from a certain species as contaminants, by providing the species name(s) in the list
- For title parsing
  - Parameter `fasta_title_regex` is `None` by default, and the title will be parsed as the UniProt format ">sp|protein|name_species ...", which will extract "protein" and "species"
  - To set a customized regex for parsing titles, this parameter can be `str` or a list of `str`
  - For example
    - `fasta_title_regex=None` -> UniProt title
    - `fasta_title_regex=r">[^|\s]+?\|(?P<protein>[^|\s]+?)\|[^\s]+?_(?P<species>[^\s]+)[$\s].*"` -> UniProt title but by regex
    - `fasta_title_regex=r">(?P<protein>[^\ ]+).+?Tax_Id=(?P<species>[^\ ]+).*?"` -> A format like ">Q32MB2 TREMBL:Q32MB2 Tax_Id=9606 Gene_Symbol=KRT73"
    - `fasta_title_regex=[regex1, regex2]` -> Try to parse title sequentially until both "protein" and "species" are found. If "protein" can not be found by any regex, will raise error. If "species" can not be found, it will be "unknown", and "protein" will be the first matched one.

In [6]:
parsed_fasta = lipana.parse_fasta(
    fasta_path=workspace.joinpath("human_yeast_contam.fasta"),
    contam_fasta_path=workspace.joinpath("contam.fasta"),
    contaminations=None,
    fasta_title_regex=None,  # Set one or more regex if the title is not UniProt format or there are mixed formats in the FASTA files
    gen_species_to_concat_seqs=True,  # This parameter is useful to annotate species for peptides via a whole protein sequence map, but is time-consuming
    workspace=workspace,
    resume=True,
    write_parsed_fasta=True,
)

### Load and annotate a search report

- Now, load and annotate the search report using the previously set up `ExperimentLayout` and `ParsedFasta`

In [7]:
# The load_search_report methods for DIA-NN and Spectronaut share parameters:
# lipana.DIANNReport.load_search_report(...)
# lipana.SpectronautReport.load_search_report(...)

# Example loading a DIA-NN search report, showing a subset of available parameters
report = lipana.DIANNReport.load_search_report(
    workspace.joinpath("example_diann_report.txt"),
    exp_layout=exp_layout,
    parsed_fasta=parsed_fasta,
    do_species_annotation=True,  # Set this to True if the experiment includes multiple species
    pre_annotation_filter=(
        (pl.col("Q.Value") < 0.01)
        & (pl.col("Lib.PG.Q.Value") < 0.01)
        & (pl.col("Protein.Group").is_not_null())
        & (pl.col("Precursor.Quantity").is_not_null())
        & (pl.col("Precursor.Quantity") > 1.1)
    ),  # The filters defined here will be applied to the report after loading immediately, and here the 5 rules are default for DIA-NN (if this parameter is not given, these rules will also be applied); for Spectronaut, they are (
    #     (pl.col("PG.ProteinGroups").is_not_null())
    #     & (pl.col("FG.Quantity").is_not_null())
    #     & (pl.col("FG.Quantity") > 1.1)
    # )
    post_annotation_filter=None,  # This parameter can be used to filter the report after annotation
    restricted_cut_sites=("K", "R"),
    expand_to_cut_site_level=True,  # This will make each cut site one row, which means each precursor row will be duplicated
    # modification_map=None,  # This parameter can be either None or a map from software used modification to UniMod. By default, DIA-NN report has None because it use UniMod as default; for `SpectronautReport`, this parameter has a default value as {
    #     "[Acetyl (Protein N-term)]": "(UniMod:1)",
    #     "[Carbamidomethyl (C)]": "(UniMod:4)",
    #     "[Oxidation (M)]": "(UniMod:35)",
    # }
)
# The variable `report` is now a `SearchReport` object, and basic annotations have been done
report

Search report object with main report shape: (113057, 81)
Number of quantification data: 0
Number of statistics results: 0
Workspace: /home/data/LiPAna/example_data

## Preparation of Quantification Matrices

- A newly created `SearchReport` object initially contains only a preprocessed search report and associated metadata, and in this section, some quantification matrices will be added to it
- The quantification process can simply involve pivoting the report or aggregating data using methods like topN or maxLFQ

### Use raw reported quantity values

- This approach converts the report into a quantification matrix by using the raw quantity values reported by the software as matrix cells. It is particularly useful for generating precursor-level or protein group-level matrices, since most software will report the quantity values on these two levels
- Additionally, this step demonstrates how to manually attach a customized quantification report to the main report using the `attach_quant_data` method

In [8]:
quant_data = report.attach_quant_data(
    lipana.convert_long_report_to_wide(
        report,
        index_col="precursor",
        column_col="run",
        value_col="precursor_quantity",
        do_log_scale=2,
        pl_filter=(
            ~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)
        ),  # This parameter filters out those peptides mapped to multiple species
        do_unique=True,
    ),
    # These two parameters will be the name of this quantification report, and can be used to retrieve it from the main report object
    # Though they can be named as any strings, it is recommended to use the same name as the entry level and the actually used quantification method
    entry_level="precursor",
    quant_method="precursor_quantity",
)

# To retrieve this object from the main report object, use the following command
quant_data = report.get_quant_data("precursor", "precursor_quantity")


### maxLFQ and topN Methods

- Both maxLFQ and topN methods utilize the same interface through the `SearchReport.construct_and_attach_quant_data` method, featuring convenient designs to handle more complex scenarios
- The following examples will concentrate on maxLFQ; topN can be configured by setting parameters like `method="top3"` or `method="top1"`, etc.


#### Peptide matrix from precursor MS2 via maxLFQ

- This example will do maxLFQ on precursor quantities to get a peptide level matrix

In [9]:
quant_data = report.construct_and_attach_quant_data(
    quant_name="peptide_quant_by_maxlfq_from_prec",  # This parameter will be the name of quantification method for this quantification report, and can be used to retrieve it from the main report object
    method="maxlfq",
    filter_condition=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)),
    run_col="run",
    primary_entry_col="stripped_peptide",  # This parameter not only define the primary (target) entry level, but also the first name to retrieve the generated matrix from the main report object
    low_level_entry_col="precursor",
    base_quant_col="precursor_quantity",
    require_expansion=False,
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)

MaxLFQ estimation via iq for /tmp/tmp12_hq_t9.txt
Removing low intensities...

Output to /tmp/tmp12_hq_t9.txt-iq_output.txt



Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmp12_hq_t9.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 55844
# quantitative values after filtering = 55844

# samples  = 9
# proteins = 9577
nrow = 55844, # proteins = 9577, # samples = 9
Using 95 threads...
0%
6%
11%
16%
21%
26%
31%
36%
41%
46%
51%
56%
62%
67%
72%
77%
82%
87%
92%
97%
Completed.


#### Peptide matrix from prec MS1 and prec MS2 via maxLFQ

- The quantities of base level entries can also be from multiple sources
- The following example will use maxLFQ to aggregate quantities from two levels: precursor MS1 and precursor MS2

In [10]:
quant_data = report.construct_and_attach_quant_data(
    quant_name="peptide_quant_by_maxlfq_from_multi",
    method="maxlfq",
    filter_condition=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)),
    run_col="run",
    primary_entry_col="stripped_peptide",
    low_level_entry_col=("precursor", "precursor"),
    base_quant_col=("precursor_quantity", "precursor_quantity_ms1"),
    require_expansion=(False, False),
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)

MaxLFQ estimation via iq for /tmp/tmpne_f0jzi.txt
Removing low intensities...




Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmpne_f0jzi.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 111685
# quantitative values after filtering = 111685

# samples  = 9
# proteins = 9577
nrow = 111685, # proteins = 9577, # samples = 9
Using 95 threads...
0%
5%
10%
15%
21%
26%
31%
37%
42%
47%
52%
57%
62%
67%
72%
77%
83%
88%
93%
98%
Completed.


Output to /tmp/tmpne_f0jzi.txt-iq_output.txt


#### Peptide matrix from prec MS1, prec MS2, and fragments, via maxLFQ

- The following example shows a more complicated case, which uses maxLFQ to aggregate quantities from three levels: precursor MS1, precursor MS2, and fragments (the fragment quantity values in the DIA-NN tsv report is in one table cell joined by ";" for each row)


In [11]:
quant_data = report.construct_and_attach_quant_data(
    quant_name="peptide_quant_by_maxlfq_from_multi",
    method="maxlfq",
    filter_condition=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)),
    run_col="run",
    primary_entry_col="stripped_peptide",
    low_level_entry_col=("precursor", "precursor", "Fragment.Info"),
    base_quant_col=("precursor_quantity", "precursor_quantity_ms1", "Fragment.Quant.Raw"),
    require_expansion=(False, False, ";"),
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)

MaxLFQ estimation via iq for /tmp/tmp1i6lqk5_.txt



Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmp1i6lqk5_.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 701484
# quantitative values after filtering = 701484

# samples  = 9
# proteins = 9577


Removing low intensities...



nrow = 701484, # proteins = 9577, # samples = 9
Using 95 threads...
0%
5%
10%
15%
21%
26%
31%
36%
41%
47%
52%
57%
62%
67%
72%
77%
82%
87%
92%
97%
Completed.


Output to /tmp/tmp1i6lqk5_.txt-iq_output.txt


#### Non-restricted cut site from precursor MS2+MS1

- Sometimes, the quantity values should be aggregated from a filtered report
- The following in-one-step example will generate a cut site level matrix, where the cut sites are only non-restricted
  - Whether a cut site is restricted has been annotated via the parameter `restricted_cut_sites` in method `load_search_report` (in this doc, it's `lipana.DIANNReport.load_search_report(..., restricted_cut_sites=("K", "R"), ...)`)
  - Besides the filtering for site annotation `(~pl.col("cut_site_is_restricted"))`, there is also a filter  for peptide, because the C-term of a cut site can be the second residue of a protein, which would be surrounded by first "M" and the third any residue. Here the cut site will be annotated as "non-restricted" if the third residue is K/R, but it's hard to say if the cleavage between protein N-term "M" and the second residue is caused during translation or by a non-restricted enzyme. Further restraining peptide to be semi-specific will address this problem by filtering out such peptides at the protein N-term.

In [12]:
quant_data = report.construct_and_attach_quant_data(
    quant_name="nonrestricted_cut_site_by_maxlfq_from_prec",
    method="maxlfq",
    filter_condition=(
        (~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))
        & (pl.col("peptide_enzymatic_specificity") == "semi_specific")
        & (~pl.col("cut_site_is_restricted"))
    ),
    run_col="run",
    primary_entry_col="cut_site",
    low_level_entry_col=("precursor", "precursor"),
    base_quant_col=("precursor_quantity", "precursor_quantity_ms1"),
    require_expansion=(False, False),
    concat_entry_after_expansion=None,
    remove_below_threshold=1.1,
    quant_input_name=None,
    attach_quant_input=False,
)

MaxLFQ estimation via iq for /tmp/tmpogkuwwf7.txt
Removing low intensities...




Command: --sample SampleIds --primary PrimaryIds --secondary AggregationIds --quant BaseQuant /tmp/tmpogkuwwf7.txt 

Sample column:
    SampleIds
Protein column:
    PrimaryIds
Ion column(s):
    AggregationIds
Quant column:
    BaseQuant

Using 4 threads ...

# lines read (excluding headers)      = 52579
# quantitative values after filtering = 52579

# samples  = 9
# proteins = 4116
nrow = 52579, # proteins = 4116, # samples = 9
Using 95 threads...
0%
5%
10%
15%
21%
26%
31%
37%
42%
47%
52%
57%
62%
68%
73%
78%
83%
88%
93%
99%
Completed.


Output to /tmp/tmpogkuwwf7.txt-iq_output.txt


### Utilities on quantification report

- The quantification matrix generated from the above sections is actually an object `EntryQuantificationReport`, which has many handful functions

In [13]:
# If the generated object is not explicitly assigned to a variable, can use the following way to get from main report object
quant_data = report.get_quant_data("precursor", "precursor_quantity")

# The following three methods will attach new columns to the dataframe stored in the quantification report object
quant_data.calc_cv(cond="all", min_reps=3, temp_reverse_log_scale=2, new_colname_pattern="{cond}_cv_{min_reps}reps")
quant_data.count_detected_replicates(new_colname_pattern="{cond}_detected_reps")
quant_data.calc_ratio(
    base_cond="H5_Y100",
    is_log=True,
    temp_reverse_log_scale=2,
    div_method="agg_and_divide",
    agg_method="mean",
    new_colname_pattern="ratio_{cond1}_to_{cond2}",
)

precursor,20240404_YSJ-HY_25_100-400ng_C_90min_stdDIA-HomeMade_Slot1-34_1_5220,20240331_YSJ-HY_5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-15_1_5158,20240331_YSJ-HY_25_100-400ng_B_90min_stdDIA-HomeMade_Slot1-20_1_5164,20240331_YSJ-HY_25_100-400ng_A_90min_stdDIA-HomeMade_Slot1-19_1_5163,20240331_YSJ-HY_5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-14_1_5157,20240404_YSJ-HY_5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-28_1_5210,20240331_YSJ-HY_2d5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-13_1_5155,20240404_YSJ-HY_2d5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-24_1_5206,20240331_YSJ-HY_2d5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-12_1_5154,H2d5_Y100_cv_3reps,H5_Y100_cv_3reps,H25_Y100_cv_3reps,H2d5_Y100_detected_reps,H5_Y100_detected_reps,H25_Y100_detected_reps,ratio_H2d5_Y100_to_H5_Y100,ratio_H25_Y100_to_H5_Y100
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,i8,i8,i8,f32,f32
"""KIIDSSSDLDK/2""",12.797358,11.797868,12.97332,11.658575,12.332917,11.99132,12.226891,12.745603,12.950882,24.360022,19.149523,41.625267,3,3,3,0.614171,0.524034
"""WVNNILYEGAESER/2""",16.704439,16.50642,16.592492,16.434168,16.39673,16.2733,16.521781,16.040816,16.230108,17.021267,8.058215,9.327686,3,3,3,-0.117363,0.185982
"""LEEIESIDQEIK/2""",null,null,15.060364,null,null,14.26982,null,null,null,NaN,NaN,NaN,0,1,1,NaN,0.790544
"""IQDLIPVLLR/2""",10.630412,null,9.864225,10.264494,null,null,null,null,null,NaN,NaN,26.177958,0,0,3,NaN,NaN
"""KNAESNAELK/2""",null,14.878989,17.26641,17.269876,14.334164,14.585806,null,null,null,NaN,18.94961,NaN,0,3,2,NaN,2.651282
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""YESVNEC(UniMod:4)MEGVC(UniMod:…",null,null,11.799745,null,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN
"""ASGSSPDAPAKDAR/2""",null,null,11.33822,null,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN
"""AGDFTNHNGTGGK/2""",null,null,null,13.450785,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN


- Because main report and quantification report are related, the annotation and filtering of quantification report can be done by borrowing data from the main report
- The following method will return a new dataframe instead of updating the dataframe stored in the object

In [14]:
report.get_quant_data(
    entry_name="precursor",
    quant_name="precursor_quantity",
    main_df_entry_filter=(
        (~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))
        & (pl.col("peptide_enzymatic_specificity") == "semi_specific")
    ),
    quant_df_filter=None,
    annotation_cols="protein_group",
)

precursor,20240404_YSJ-HY_25_100-400ng_C_90min_stdDIA-HomeMade_Slot1-34_1_5220,20240331_YSJ-HY_5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-15_1_5158,20240331_YSJ-HY_25_100-400ng_B_90min_stdDIA-HomeMade_Slot1-20_1_5164,20240331_YSJ-HY_25_100-400ng_A_90min_stdDIA-HomeMade_Slot1-19_1_5163,20240331_YSJ-HY_5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-14_1_5157,20240404_YSJ-HY_5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-28_1_5210,20240331_YSJ-HY_2d5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-13_1_5155,20240404_YSJ-HY_2d5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-24_1_5206,20240331_YSJ-HY_2d5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-12_1_5154,H2d5_Y100_cv_3reps,H5_Y100_cv_3reps,H25_Y100_cv_3reps,H2d5_Y100_detected_reps,H5_Y100_detected_reps,H25_Y100_detected_reps,ratio_H2d5_Y100_to_H5_Y100,ratio_H25_Y100_to_H5_Y100,protein_group
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,i8,i8,i8,f32,f32,str
"""KIIDSSSDLDK/2""",12.797358,11.797868,12.97332,11.658575,12.332917,11.99132,12.226891,12.745603,12.950882,24.360022,19.149523,41.625267,3,3,3,0.614171,0.524034,"""P26637"""
"""LEEIESIDQEIK/2""",null,null,15.060364,null,null,14.26982,null,null,null,NaN,NaN,NaN,0,1,1,NaN,0.790544,"""P78318"""
"""KNAESNAELK/2""",null,14.878989,17.26641,17.269876,14.334164,14.585806,null,null,null,NaN,18.94961,NaN,0,3,2,NaN,2.651282,"""P18621"""
"""HMDGGQIDGQEITAT/2""",16.792723,13.887778,16.335333,16.279752,13.507572,13.100761,12.399578,12.196999,12.071223,11.61167,26.908682,20.425512,3,3,3,-1.305178,2.954003,"""Q15287"""
"""YPAVDPLDSK/2""",14.256015,14.276068,14.5218,14.318231,14.467694,14.040767,14.502536,null,14.393954,NaN,14.67044,9.830509,2,3,3,0.177264,0.097872,"""P00830"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""YESVNEC(UniMod:4)MEGVC(UniMod:…",null,null,11.799745,null,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN,"""P84090"""
"""ASGSSPDAPAKDAR/2""",null,null,11.33822,null,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN,"""O75962"""
"""AGDFTNHNGTGGK/2""",null,null,null,13.450785,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN,"""P30405"""


In [15]:
quant_data = report.get_quant_data("precursor", "precursor_quantity")
quant_data.attach_annotation_via_entry(
    "protein_group", persistent=False
)  # Set `persistent` to True will update the dataframe stored in the object
quant_data.filter_entry_by_main_report(
    main_report_filter=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))
    & (pl.col("peptide_enzymatic_specificity") == "semi_specific")
)


precursor,20240404_YSJ-HY_25_100-400ng_C_90min_stdDIA-HomeMade_Slot1-34_1_5220,20240331_YSJ-HY_5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-15_1_5158,20240331_YSJ-HY_25_100-400ng_B_90min_stdDIA-HomeMade_Slot1-20_1_5164,20240331_YSJ-HY_25_100-400ng_A_90min_stdDIA-HomeMade_Slot1-19_1_5163,20240331_YSJ-HY_5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-14_1_5157,20240404_YSJ-HY_5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-28_1_5210,20240331_YSJ-HY_2d5_100-400ng_B_90min_stdDIA-HomeMade_Slot1-13_1_5155,20240404_YSJ-HY_2d5_100-400ng_C_90min_stdDIA-HomeMade_Slot1-24_1_5206,20240331_YSJ-HY_2d5_100-400ng_A_90min_stdDIA-HomeMade_Slot1-12_1_5154,H2d5_Y100_cv_3reps,H5_Y100_cv_3reps,H25_Y100_cv_3reps,H2d5_Y100_detected_reps,H5_Y100_detected_reps,H25_Y100_detected_reps,ratio_H2d5_Y100_to_H5_Y100,ratio_H25_Y100_to_H5_Y100
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,i8,i8,i8,f32,f32
"""KIIDSSSDLDK/2""",12.797358,11.797868,12.97332,11.658575,12.332917,11.99132,12.226891,12.745603,12.950882,24.360022,19.149523,41.625267,3,3,3,0.614171,0.524034
"""LEEIESIDQEIK/2""",null,null,15.060364,null,null,14.26982,null,null,null,NaN,NaN,NaN,0,1,1,NaN,0.790544
"""KNAESNAELK/2""",null,14.878989,17.26641,17.269876,14.334164,14.585806,null,null,null,NaN,18.94961,NaN,0,3,2,NaN,2.651282
"""HMDGGQIDGQEITAT/2""",16.792723,13.887778,16.335333,16.279752,13.507572,13.100761,12.399578,12.196999,12.071223,11.61167,26.908682,20.425512,3,3,3,-1.305178,2.954003
"""YPAVDPLDSK/2""",14.256015,14.276068,14.5218,14.318231,14.467694,14.040767,14.502536,null,14.393954,NaN,14.67044,9.830509,2,3,3,0.177264,0.097872
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""YESVNEC(UniMod:4)MEGVC(UniMod:…",null,null,11.799745,null,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN
"""ASGSSPDAPAKDAR/2""",null,null,11.33822,null,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN
"""AGDFTNHNGTGGK/2""",null,null,null,13.450785,null,null,null,null,null,NaN,NaN,NaN,0,0,1,NaN,NaN


## Statistics

- The `lipana` package provides several built-in functions for statistical analysis, allowing users to customize their statistical pipeline by chaining these functions together. Additionally, there is a packed function containing three predefined pipelines
- This section will demonstrate how to use the predefined pipelines for statistical tests
- Note: function `lipana.do_stats_pipeline_pairwise` includes an optional parameter `missing_value_config`, which defaults to `None`
  - When not specified or set to `None`, it is initialized as `FullEmptyFillingMissingValueHandler(min_rep_count=2)`, which fills the fully empty condition as 0 (log-scale) for an entry if the paired condition has enough detected replicates (>=`min_rep_count`)
  - To disable missing value filling, set this parameter to `lipana.NullMissingValueHandler()`

### Select minimum p-value

- This pipeline performs tests at the base entry level and selects test results using the minimum p-value among associated base entries for each target entry
- For example, when testing at the precursor level, each protein may have multiple precursors. In this case, for one protein, the fold change (FC), p-value, and other metrics will match those of the precursor with the minimum p-value among all associated precursors
- Note: this pipeline will do a pre-filtering based on the sign of t-statistic
  - When there is only one row for a target entry, this row will be kept
  - If two rows have different sign values, drop this target entry
  - If greater than or equal to three rows, first determine the sign value that more than half rows have, and select one row with min p-value from the rows having that sign value

- Below shows an example to select precursor with min p-value for cut site
  - In this case, the report should be first filtered to only keep those non-restricted cut site (i.e. non KR as defined when loading the report)

In [16]:
design = lipana.ComparisonDesign(exp_layout, (("H2d5_Y100", "H5_Y100"), ("H25_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

filters = ~pl.col("cut_site_is_restricted") & (~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))

stats = lipana.do_stats_pipeline_pairwise(
    qdf=report.get_quant_data(
        entry_name="precursor",
        quant_name="precursor_quantity",
        main_df_entry_filter=filters,
    ),  # Note here no need to call `.df` because the requirement of filtering will make this function return a dataframe, instead of a `EntryQuantificationReport` object
    main_df=report.df.filter(filters),
    design=design,
    target_entry_col="cut_site",
    base_entry_col="precursor",
    group_entry_col="protein_group",  # Note this parameter is given because FDR control also can be done on the protein level
    pipeline="sel_min_p",
    return_chains=False,
)
sleep(0.1)
print(f"First 5 rows of the result (total size: {stats.shape}):")
stats.head(5)

design.pairwise_comparisons = (('H2d5_Y100', 'H5_Y100'), ('H25_Y100', 'H5_Y100'))


Quantification file: /tmp/tmphvpxkuk8.txt
All defined comparison pairs: c("H2d5_Y100", "H5_Y100")
Available comparison pairs: c("H2d5_Y100", "H5_Y100")
Comparing H2d5_Y100 vs H5_Y100
Output to /tmp/tmphvpxkuk8.txt-limma_output.txt
Quantification file: /tmp/tmp37pdtkef.txt
All defined comparison pairs: c("H25_Y100", "H5_Y100")
Available comparison pairs: c("H25_Y100", "H5_Y100")
Comparing H25_Y100 vs H5_Y100
Output to /tmp/tmp37pdtkef.txt-limma_output.txt


First 5 rows of the result (total size: (4992, 15)):


precursor,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,cut_site,row_sign,group_sign,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise
str,f64,f64,f64,f64,f64,f64,str,f32,str,f64,f64,f64,str,f64
"""KQVEQLEK/2""",-15.397284,7.698642,-96.296646,3.3869e-13,2.4894e-10,16.855227,"""H2d5_Y100_vs_H5_Y100""",-15.400756,"""Q14152-671_672""",-1.0,-1.0,2.1368e-10,"""Q14152""",6.0964e-12
"""ADHQPLTEA/2""",-14.391438,7.195719,-93.03139,4.4224e-13,2.4894e-10,16.787351,"""H2d5_Y100_vs_H5_Y100""",-14.392776,"""P08865-137_138""",-1.0,-1.0,2.1368e-10,"""P08865""",3.6495e-12
"""LFPEQQTK/2""",14.187453,7.093726,92.570853,4.5954e-13,2.4894e-10,16.777286,"""H2d5_Y100_vs_H5_Y100""",14.188194,"""P05739-120_121""",1.0,1.0,2.1368e-10,"""P05739""",4.5954e-13
"""YDSVEEGEKVVK/2""",-16.438677,8.219339,-89.852936,5.7863e-13,2.4894e-10,16.71522,"""H2d5_Y100_vs_H5_Y100""",-16.452827,"""P51659-72_73""",-1.0,-1.0,2.1368e-10,"""P51659""",2.8931e-12
"""FTPGTFTNQIQAA/2""",-14.546217,7.273109,-89.274668,6.0825e-13,2.4894e-10,16.7014,"""H2d5_Y100_vs_H5_Y100""",-14.550974,"""P08865-115_116""",-1.0,-1.0,2.1368e-10,"""P08865""",3.6495e-12


### Select minimum p-value from all

- This pipeline is similar as the above one, and only differ in the absence of the pre-filtering step, which means one row with the minimum p-value will be selected for a target entry, despite what the sign of t-statistic it has
  - Set `pipeline="sel_min_p_from_all"` to use this pipeline
- Below is an example to select precursor with min p-value for protein, without pre-filtering of precursors

In [17]:
design = lipana.ComparisonDesign(exp_layout, (("H2d5_Y100", "H5_Y100"), ("H25_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

stats = lipana.do_stats_pipeline_pairwise(
    qdf=report.get_quant_data(entry_name="precursor", quant_name="precursor_quantity").df,
    main_df=report.df,
    design=design,
    target_entry_col="protein_group",
    base_entry_col="precursor",
    group_entry_col=None,
    pipeline="sel_min_p_from_all",
    return_chains=False,
)
sleep(0.1)
print(f"First 5 rows of the result (total size: {stats.shape}):")
stats.head(5)

design.pairwise_comparisons = (('H2d5_Y100', 'H5_Y100'), ('H25_Y100', 'H5_Y100'))


Quantification file: /tmp/tmp0b8sy23s.txt
All defined comparison pairs: c("H2d5_Y100", "H5_Y100")
Available comparison pairs: c("H2d5_Y100", "H5_Y100")
Comparing H2d5_Y100 vs H5_Y100
Output to /tmp/tmp0b8sy23s.txt-limma_output.txt
Quantification file: /tmp/tmpfdl2zrw2.txt
All defined comparison pairs: c("H25_Y100", "H5_Y100")
Available comparison pairs: c("H25_Y100", "H5_Y100")
Comparing H25_Y100 vs H5_Y100
Output to /tmp/tmpfdl2zrw2.txt-limma_output.txt


First 5 rows of the result (total size: (1496, 11)):


precursor,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,protein_group,adj_pvalue_exp_wise
str,f64,f64,f64,f64,f64,f64,str,f32,str,f64
"""KQVEQLEK/2""",-15.397284,7.698642,-102.662345,1.0859e-12,7.0257e-10,15.773698,"""H2d5_Y100_vs_H5_Y100""",-15.400756,"""Q14152""",1.0261e-10
"""ADHQPLTEA/2""",-14.391438,7.195719,-99.930526,1.3191e-12,7.0257e-10,15.728703,"""H2d5_Y100_vs_H5_Y100""",-14.392776,"""P08865""",1.0261e-10
"""LFPEQQTK/2""",14.187453,7.093726,99.658141,1.3453e-12,7.0257e-10,15.724041,"""H2d5_Y100_vs_H5_Y100""",14.188194,"""P05739""",1.0261e-10
"""HLEMNPHFGSHR/3""",-13.921952,6.960976,-97.462274,1.5799e-12,7.0257e-10,15.685227,"""H2d5_Y100_vs_H5_Y100""",-13.922854,"""Q08211""",1.0261e-10
"""RLEC(UniMod:4)GFPDYK/2""",-13.815381,6.90769,-97.269656,1.6026e-12,7.0257e-10,15.681715,"""H2d5_Y100_vs_H5_Y100""",-13.815996,"""Q3E793""",1.0261e-10


### Combine p-values

- This pipeline tests at the base entry level, aggregates fold change values using the median, and combines p-values using Fisher's combined probability test
- The API for this method is similar to the "select minimum p-value" method, but differs in one parameter
- Here, an example will be shown for testing peptides and aggregating results to the cut site level

In [18]:
design = lipana.ComparisonDesign(exp_layout, (("H2d5_Y100", "H5_Y100"), ("H25_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

filters = ~pl.col("cut_site_is_restricted") & (~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))

stats = lipana.do_stats_pipeline_pairwise(
    qdf=report.get_quant_data(
        entry_name="stripped_peptide",
        quant_name="peptide_quant_by_maxlfq_from_prec",  # this report is generated in one previous cell
        main_df_entry_filter=filters,
    ),  # Note here no need to call `.df` because the requirement of filtering will make this function return a dataframe, instead of a `EntryQuantificationReport` object
    main_df=report.df.filter(filters),
    design=design,
    target_entry_col="cut_site",
    base_entry_col="stripped_peptide",
    group_entry_col="protein_group",  # Note this parameter is given because FDR control also can be done on the protein level
    pipeline="combine_p",  # Note this parameter is different from that in the previous example
    return_chains=False,
)
sleep(0.1)
print(f"First 5 rows of the result (total size: {stats.shape}):")
stats.head(5)

design.pairwise_comparisons = (('H2d5_Y100', 'H5_Y100'), ('H25_Y100', 'H5_Y100'))


Quantification file: /tmp/tmpjbrv6p14.txt
All defined comparison pairs: c("H2d5_Y100", "H5_Y100")
Available comparison pairs: c("H2d5_Y100", "H5_Y100")
Comparing H2d5_Y100 vs H5_Y100
Output to /tmp/tmpjbrv6p14.txt-limma_output.txt
Quantification file: /tmp/tmphrbi659x.txt
All defined comparison pairs: c("H25_Y100", "H5_Y100")
Available comparison pairs: c("H25_Y100", "H5_Y100")
Comparing H25_Y100 vs H5_Y100
Output to /tmp/tmphrbi659x.txt-limma_output.txt


First 5 rows of the result (total size: (5077, 9)):


cut_site,log2_fc,log2_fc_limma,pvalue,adj_pvalue_limma,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise,pair
str,f32,f64,f64,f64,f64,str,f64,str
"""Q15427-101_102""",-12.757358,-12.75668,2.7047e-9,4.8096e-8,4.2172e-8,"""Q15427""",2.7047e-9,"""H2d5_Y100_vs_H5_Y100"""
"""P15992-22_23""",-0.138497,-0.128327,0.495373,0.901235,0.871373,"""P15992""",0.888301,"""H2d5_Y100_vs_H5_Y100"""
"""P84090-1_2""",-1.253025,-1.276497,0.000311,0.001798,0.001598,"""P84090""",0.000466,"""H2d5_Y100_vs_H5_Y100"""
"""P38708-423_424""",0.151186,0.161867,0.400263,0.832622,0.797661,"""P38708""",0.998155,"""H2d5_Y100_vs_H5_Y100"""
"""P50454-324_325""",-13.837998,-13.837411,1.6792e-9,3.6414e-8,3.1568e-8,"""P50454""",3.3585e-9,"""H2d5_Y100_vs_H5_Y100"""


### Direct tests on target level

- This method performs tests directly on the expected target entry level, and the input must be a quantification matrix at this level
- An example will be provided for testing fully-specific stripped peptides

In [19]:
design = lipana.ComparisonDesign(exp_layout, (("H2d5_Y100", "H5_Y100"), ("H25_Y100", "H5_Y100")))
print(f"{design.pairwise_comparisons = }")

filters = (pl.col("peptide_enzymatic_specificity") == "fully_specific") & (
    ~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)
)

stats = lipana.do_stats_pipeline_pairwise(
    qdf=report.get_quant_data(
        entry_name="stripped_peptide",
        quant_name="peptide_quant_by_maxlfq_from_prec",  # this report is generated in one previous cell
        main_df_entry_filter=filters,
    ),
    main_df=report.df.filter(filters),
    design=design,
    target_entry_col="stripped_peptide",
    base_entry_col=None,  # Note in this case, `base_entry_col` should be None
    group_entry_col="protein_group",
    pipeline="direct_test",  # Note this parameter is different from that in the previous example
    return_chains=False,
)
sleep(0.1)
print(f"First 5 rows of the result (total size: {stats.shape}):")
stats.head(5)

design.pairwise_comparisons = (('H2d5_Y100', 'H5_Y100'), ('H25_Y100', 'H5_Y100'))


Quantification file: /tmp/tmpb_8nmr7x.txt
All defined comparison pairs: c("H2d5_Y100", "H5_Y100")
Available comparison pairs: c("H2d5_Y100", "H5_Y100")
Comparing H2d5_Y100 vs H5_Y100
Output to /tmp/tmpb_8nmr7x.txt-limma_output.txt
Quantification file: /tmp/tmp0wgp1wh6.txt
All defined comparison pairs: c("H25_Y100", "H5_Y100")
Available comparison pairs: c("H25_Y100", "H5_Y100")
Comparing H25_Y100 vs H5_Y100
Output to /tmp/tmp0wgp1wh6.txt-limma_output.txt


First 5 rows of the result (total size: (6345, 12)):


stripped_peptide,log2_fc_limma,AveExpr,t,pvalue,adj_pvalue_limma,B,pair,log2_fc,adj_pvalue_exp_wise,protein_group,adj_pvalue_group_wise
str,f64,f64,f64,f64,f64,f64,str,f32,f64,str,f64
"""RLECGFPDYK""",-13.815381,6.90769,-104.018898,1.9408e-12,7.6420e-10,15.496262,"""H2d5_Y100_vs_H5_Y100""",-13.815996,7.6420e-10,"""Q3E793""",3.8816e-12
"""EIYHFTLEK""",-14.353165,7.176582,-103.074231,2.0689e-12,7.6420e-10,15.481102,"""H2d5_Y100_vs_H5_Y100""",-14.355913,7.6420e-10,"""Q9BT78""",1.0344e-11
"""QWGWTQGR""",-14.61908,7.30954,-100.38024,2.4906e-12,7.6420e-10,15.435839,"""H2d5_Y100_vs_H5_Y100""",-14.624066,7.6420e-10,"""P18621""",1.4944e-11
"""TDGKVFQFLNAK""",-14.149595,7.074797,-100.225904,2.5176e-12,7.6420e-10,15.433152,"""H2d5_Y100_vs_H5_Y100""",-14.153047,7.6420e-10,"""P83731""",1.0070e-11
"""SADLYKDR""",-13.151591,6.575795,-98.236877,2.8971e-12,7.6420e-10,15.397541,"""H2d5_Y100_vs_H5_Y100""",-13.152547,7.6420e-10,"""Q9HBM1""",2.8971e-12
